<a href="https://colab.research.google.com/github/dontaedawkins9-ux/IS-362/blob/main/IS362_Project2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IS 362 – Project 2

## Tidying and Transforming Wide Datasets

The goal of this project is to practice preparing untidy or "wide" datasets for analysis using Python and pandas.

For this project, I will work with three different datasets. For each dataset, I will:

1. Load the original wide-format data from a CSV file.
2. Examine the structure of the dataset.
3. Tidy and transform the data using pandas.
4. Perform an analysis on the cleaned data.
5. Describe the results and conclusions.

### Datasets
1. NFL Quarterback Statistics
2. Video Game Sales
3. Movie Box Office Data

In [1]:
import pandas as pd

# Create the original wide NFL quarterback dataset
nfl_data = {
    "Quarterback": ["Patrick Mahomes", "Josh Allen", "Lamar Jackson"],
    "Pass Yards 2023": [4183, 4306, 3678],
    "Pass Yards 2024": [3928, 3731, 4172],
    "Pass Yards 2025": [4587, 3668, 4050],
    "TDs 2023": [27, 29, 24],
    "TDs 2024": [26, 28, 41],
    "TDs 2025": [30, 25, 34]
}

nfl_wide = pd.DataFrame(nfl_data)

# Save the wide dataset as a CSV file
nfl_wide.to_csv("nfl_quarterback_stats.csv", index=False)

nfl_wide

,Quarterback,Pass Yards 2023,Pass Yards 2024,Pass Yards 2025,TDs 2023,TDs 2024,TDs 2025
0,Patrick Mahomes,4183,3928,4587,27,26,30
1,Josh Allen,4306,3731,3668,29,28,25
2,Lamar Jackson,3678,4172,4050,24,41,34


In [2]:
# Read the wide NFL dataset from the CSV file
nfl = pd.read_csv("nfl_quarterback_stats.csv")

# Display the original untidy dataset
nfl

,Quarterback,Pass Yards 2023,Pass Yards 2024,Pass Yards 2025,TDs 2023,TDs 2024,TDs 2025
0,Patrick Mahomes,4183,3928,4587,27,26,30
1,Josh Allen,4306,3731,3668,29,28,25
2,Lamar Jackson,3678,4172,4050,24,41,34


## Dataset 1: NFL Quarterback Statistics

### Original Data

The original NFL dataset is in a wide format. Each quarterback has one row, while the season and type of statistic are included in the column names. For example, "Pass Yards 2023" and "Pass Yards 2024" represent the same variable for different seasons.

This structure is considered untidy because the year is stored as part of the column name instead of being its own variable. To make the dataset easier to analyze, I will transform it so that Quarterback, Season, Passing Yards, and Touchdowns are separate variables.

In [3]:
# Transform passing yards from wide to long format
yards = nfl.melt(
    id_vars=["Quarterback"],
    value_vars=["Pass Yards 2023", "Pass Yards 2024", "Pass Yards 2025"],
    var_name="Statistic_Season",
    value_name="Passing_Yards"
)

# Extract the season from the column name
yards["Season"] = yards["Statistic_Season"].str.extract(r"(\d{4})")

# Transform touchdowns from wide to long format
touchdowns = nfl.melt(
    id_vars=["Quarterback"],
    value_vars=["TDs 2023", "TDs 2024", "TDs 2025"],
    var_name="Statistic_Season",
    value_name="Touchdowns"
)

# Extract the season
touchdowns["Season"] = touchdowns["Statistic_Season"].str.extract(r"(\d{4})")

# Combine passing yards and touchdowns into one tidy dataset
nfl_tidy = pd.merge(
    yards[["Quarterback", "Season", "Passing_Yards"]],
    touchdowns[["Quarterback", "Season", "Touchdowns"]],
    on=["Quarterback", "Season"]
)

nfl_tidy

,Quarterback,Season,Passing_Yards,Touchdowns
0,Patrick Mahomes,2023,4183,27
1,Josh Allen,2023,4306,29
2,Lamar Jackson,2023,3678,24
3,Patrick Mahomes,2024,3928,26
4,Josh Allen,2024,3731,28
5,Lamar Jackson,2024,4172,41
6,Patrick Mahomes,2025,4587,30
7,Josh Allen,2025,3668,25
8,Lamar Jackson,2025,4050,34


### Data Cleanup and Transformation

I used pandas to transform the original wide dataset into a tidy format. I first used the `melt()` function to convert the passing-yard columns into rows and extracted the season from each original column name.

I repeated this process for the touchdown columns. I then merged the passing-yard and touchdown data using the quarterback and season as matching variables.

The final tidy dataset contains four variables: Quarterback, Season, Passing_Yards, and Touchdowns. Each row now represents one quarterback's statistics for one season. This structure makes it easier to compare player performance across multiple seasons.

In [4]:
# Compare quarterback performance across the three seasons
nfl_analysis = nfl_tidy.sort_values(
    by=["Season", "Passing_Yards"],
    ascending=[True, False]
)

nfl_analysis

,Quarterback,Season,Passing_Yards,Touchdowns
1,Josh Allen,2023,4306,29
0,Patrick Mahomes,2023,4183,27
2,Lamar Jackson,2023,3678,24
5,Lamar Jackson,2024,4172,41
3,Patrick Mahomes,2024,3928,26
4,Josh Allen,2024,3731,28
6,Patrick Mahomes,2025,4587,30
8,Lamar Jackson,2025,4050,34
7,Josh Allen,2025,3668,25


In [5]:
# Calculate each quarterback's average passing yards and touchdowns
qb_averages = nfl_tidy.groupby("Quarterback")[["Passing_Yards", "Touchdowns"]].mean()

qb_averages.sort_values(by="Passing_Yards", ascending=False)

,Passing_Yards,Touchdowns
Quarterback,,
Patrick Mahomes,4232.666667,27.666667
Lamar Jackson,3966.666667,33.000000
Josh Allen,3901.666667,27.333333


### Analysis and Conclusion

After transforming the dataset, I compared the quarterbacks' passing yards and touchdowns across the 2023, 2024, and 2025 seasons.

The results show that each quarterback's performance changed from season to season. Josh Allen had the most passing yards among the three quarterbacks in 2023, Lamar Jackson had the most in 2024, and Patrick Mahomes had the most in 2025.

I also calculated the average passing yards and touchdowns for each quarterback across all three seasons. This analysis was easier to perform after converting the original wide dataset into a tidy format where each quarterback and season had its own row.

Overall, the transformation made it easier to group, sort, and compare the quarterback statistics across multiple seasons.

## Dataset 2: Video Game Sales

For my second dataset, I will use video game sales data. The original dataset will use a wide format, with sales from different regions stored in separate columns for each year.

The goal will be to tidy and transform the dataset so that the year and region become separate variables. After cleaning the data, I will compare sales across games, regions, and years.

In [6]:
# Create the original wide video game sales dataset
game_data = {
    "Game": ["Minecraft", "Grand Theft Auto V", "Mario Kart 8"],
    "NA Sales 2023": [8.2, 6.5, 5.8],
    "NA Sales 2024": [9.1, 6.9, 6.4],
    "EU Sales 2023": [7.5, 5.9, 4.9],
    "EU Sales 2024": [8.0, 6.2, 5.5]
}

games_wide = pd.DataFrame(game_data)

# Save the wide dataset as a CSV file
games_wide.to_csv("video_game_sales.csv", index=False)

games_wide

,Game,NA Sales 2023,NA Sales 2024,EU Sales 2023,EU Sales 2024
0,Minecraft,8.2,9.1,7.5,8.0
1,Grand Theft Auto V,6.5,6.9,5.9,6.2
2,Mario Kart 8,5.8,6.4,4.9,5.5


In [7]:
# Read the wide video game sales dataset from the CSV file
games = pd.read_csv("video_game_sales.csv")

# Display the original untidy dataset
games

,Game,NA Sales 2023,NA Sales 2024,EU Sales 2023,EU Sales 2024
0,Minecraft,8.2,9.1,7.5,8.0
1,Grand Theft Auto V,6.5,6.9,5.9,6.2
2,Mario Kart 8,5.8,6.4,4.9,5.5


In [8]:
# Transform the regional sales columns from wide to long format
games_tidy = games.melt(
    id_vars=["Game"],
    var_name="Region_Year",
    value_name="Sales"
)

# Separate Region and Year into their own columns
games_tidy[["Region", "Year"]] = games_tidy["Region_Year"].str.extract(
    r"(NA|EU) Sales (\d{4})"
)

# Remove the original combined column
games_tidy = games_tidy.drop(columns=["Region_Year"])

# Arrange the columns
games_tidy = games_tidy[["Game", "Year", "Region", "Sales"]]

games_tidy

,Game,Year,Region,Sales
0,Minecraft,2023,NA,8.2
1,Grand Theft Auto V,2023,NA,6.5
2,Mario Kart 8,2023,NA,5.8
3,Minecraft,2024,NA,9.1
4,Grand Theft Auto V,2024,NA,6.9
5,Mario Kart 8,2024,NA,6.4
6,Minecraft,2023,EU,7.5
7,Grand Theft Auto V,2023,EU,5.9
8,Mario Kart 8,2023,EU,4.9
9,Minecraft,2024,EU,8.0


### Data Cleanup and Transformation

The original video game dataset was in a wide format. The region and year were stored together in column names such as "NA Sales 2023" and "EU Sales 2024."

I used the pandas `melt()` function to convert these sales columns into rows. I then separated the region and year into their own columns and removed the original combined column.

The final tidy dataset contains four variables: Game, Year, Region, and Sales. Each row now represents the sales for one game in one region during one year. This format makes it easier to compare sales between games, regions, and years.

In [9]:
# Compare total sales for each game
game_totals = games_tidy.groupby("Game")["Sales"].sum().sort_values(ascending=False)

game_totals

,Sales
Game,
Minecraft,32.8
Grand Theft Auto V,25.5
Mario Kart 8,22.6


In [10]:
# Compare total sales by region and year
region_year_sales = games_tidy.groupby(
    ["Year", "Region"]
)["Sales"].sum()

region_year_sales

Year  Region
2023  EU        18.3
      NA        20.5
2024  EU        19.7
      NA        22.4
Name: Sales, dtype: float64

### Analysis and Conclusion

After transforming the video game dataset into a tidy format, I compared the total sales for each game and also compared sales by region and year.

Based on the dataset, Minecraft had the highest combined sales among the three games. Grand Theft Auto V had the second-highest total, followed by Mario Kart 8.

I also compared North American (NA) and European (EU) sales for 2023 and 2024. The data shows how sales varied by region and how they changed from one year to the next.

Converting the original wide dataset into a tidy format made it easier to group the data by game, year, and region and perform these comparisons using pandas.

## Dataset 3: Movie Box Office Data

For my third dataset, I will use movie box office data. The original dataset will be in a wide format, with domestic and international box office revenue from different years stored in separate columns.

The goal is to transform the data so that year and market become separate variables. After tidying the data, I will compare the box office performance of the movies across markets and years.

In [11]:
# Create the original wide movie box office dataset
movie_data = {
    "Movie": ["Movie A", "Movie B", "Movie C"],
    "Domestic 2023": [350, 280, 225],
    "Domestic 2024": [390, 310, 260],
    "International 2023": [500, 420, 350],
    "International 2024": [550, 460, 390]
}

movies_wide = pd.DataFrame(movie_data)

# Save the wide dataset as a CSV file
movies_wide.to_csv("movie_box_office.csv", index=False)

movies_wide

,Movie,Domestic 2023,Domestic 2024,International 2023,International 2024
0,Movie A,350,390,500,550
1,Movie B,280,310,420,460
2,Movie C,225,260,350,390


In [12]:
# Read the wide movie box office dataset from the CSV file
movies = pd.read_csv("movie_box_office.csv")

# Display the original untidy dataset
movies

,Movie,Domestic 2023,Domestic 2024,International 2023,International 2024
0,Movie A,350,390,500,550
1,Movie B,280,310,420,460
2,Movie C,225,260,350,390


In [13]:
# Transform the movie box office columns from wide to long format
movies_tidy = movies.melt(
    id_vars=["Movie"],
    var_name="Market_Year",
    value_name="Revenue"
)

# Separate Market and Year into their own columns
movies_tidy[["Market", "Year"]] = movies_tidy["Market_Year"].str.extract(
    r"(Domestic|International) (\d{4})"
)

# Remove the original combined column
movies_tidy = movies_tidy.drop(columns=["Market_Year"])

# Arrange the columns
movies_tidy = movies_tidy[["Movie", "Year", "Market", "Revenue"]]

movies_tidy

,Movie,Year,Market,Revenue
0,Movie A,2023,Domestic,350
1,Movie B,2023,Domestic,280
2,Movie C,2023,Domestic,225
3,Movie A,2024,Domestic,390
4,Movie B,2024,Domestic,310
5,Movie C,2024,Domestic,260
6,Movie A,2023,International,500
7,Movie B,2023,International,420
8,Movie C,2023,International,350
9,Movie A,2024,International,550


### Data Cleanup and Transformation

The original movie box office dataset was in a wide format. The market and year were combined in column names such as "Domestic 2023" and "International 2024."

I used the pandas `melt()` function to convert these columns into rows. I then separated the market and year information into their own columns and removed the original combined column.

The final tidy dataset contains four variables: Movie, Year, Market, and Revenue. Each row now represents the revenue for one movie in one market during one year. This structure makes it easier to compare box office revenue across movies, markets, and years.

In [14]:
# Compare total box office revenue for each movie
movie_totals = movies_tidy.groupby("Movie")["Revenue"].sum().sort_values(ascending=False)

movie_totals

,Revenue
Movie,
Movie A,1790
Movie B,1470
Movie C,1225


In [15]:
# Compare box office revenue by market and year
market_year_revenue = movies_tidy.groupby(
    ["Year", "Market"]
)["Revenue"].sum()

market_year_revenue

Year  Market       
2023  Domestic          855
      International    1270
2024  Domestic          960
      International    1400
Name: Revenue, dtype: int64

### Analysis and Conclusion

After transforming the movie box office dataset into a tidy format, I compared the total revenue for each movie and also examined revenue by market and year.

Movie A had the highest combined box office revenue, followed by Movie B and Movie C. The data also showed that international revenue was higher than domestic revenue for the movies in both years.

Transforming the original wide dataset made it easier to group the data by movie, year, and market and compare box office performance using pandas.

## Overall Conclusion

This project demonstrated how wide datasets can be transformed into tidy datasets using Python and pandas. In all three datasets, information such as year, region, statistic, or market was originally stored inside column names.

I used pandas functions such as `melt()`, `str.extract()`, `drop()`, `merge()`, `groupby()`, and `sort_values()` to reorganize and analyze the data.

After tidying the datasets, each observation had its own row and each variable had its own column. This made it easier to perform comparisons and analyze the NFL quarterback statistics, video game sales, and movie box office data.